<a href="https://colab.research.google.com/github/Nyauntu/Data-science-practice/blob/main/kiva_api_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/kiva', exist_ok=True)
os.chdir('/content/drive/MyDrive/kiva')
print('Saving to:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/kiva


In [5]:
import requests, pandas as pd, time, json, os, glob

URL = 'https://gateway.production.kiva.org/graphql'
CAP = 20000
FIELDS = '''id loanAmount status use fundraisingDate raisedDate gender borrowerCount
            lenderRepaymentTerm activity { name } sector { name } geocode { city }
            ... on LoanPartner { partnerName disbursalDate }'''

# the four slices that were too big
BIG = [dict(gender='female', isGroup=False, term=(14, 14)),
       dict(gender='male',   isGroup=False, term=(14, 14)),
       dict(gender='female', isGroup=False, term=(8, 8)),
       dict(gender='female', isGroup=False, term=(20, 20))]
# extra ways to split, each a range of ID numbers
DIMS = [('sector', 1, 60), ('activity', 1, 400), ('partner', 1, 2500)]

def gql(q):
    for a in range(6):
        try:
            r = requests.post(URL, json={'query': q}, timeout=90).json()
            if 'errors' in r: raise RuntimeError(r['errors'])
            return r['data']
        except Exception as e:
            print('   retry:', str(e)[:200]); time.sleep(5 * (a + 1))
    raise SystemExit('Giving up on query:\n' + q)

def fstr(f):
    p = ['country: ["KE"]', 'status: all',
         f"gender: {f['gender']}", f"isGroup: {str(f['isGroup']).lower()}",
         f"lenderTerm: {{min: {f['term'][0]}, max: {f['term'][1]}}}"]
    for dim, _, _ in DIMS:
        if dim in f:
            lo, hi = f[dim]
            p.append(f"{dim}: {list(range(lo, hi + 1))}")
    return '{' + ', '.join(p) + '}'

def label(f):
    return ', '.join(f'{k}={v}' for k, v in f.items())

def count(f):
    q = f'{{ lend {{ loans(filters: {fstr(f)}, limit: 1) {{ totalCount }} }} }}'
    return gql(q)['lend']['loans']['totalCount']

def split(f):
    for dim, lo0, hi0 in DIMS:
        if dim not in f:
            return [dict(f, **{dim: (lo0, hi0)})]          # add this dimension
        lo, hi = f[dim]
        if hi > lo:
            mid = (lo + hi) // 2
            return [dict(f, **{dim: (lo, mid)}), dict(f, **{dim: (mid + 1, hi)})]
    return None

def plan(f, n=None, depth=0):
    n = count(f) if n is None else n
    print('  ' * depth + f'{label(f)} -> {n:,}')
    if n <= CAP: return [(f, n)]
    kids = split(f)
    if kids is None:
        print('  ' * depth + '   !! cannot split further'); return [(f, n)]
    counts = [count(k) for k in kids]
    if sum(counts) < n:
        print('  ' * depth + f'   !! this split covers {sum(counts):,} of {n:,}')
    out = []
    for k, c in zip(kids, counts):
        if c > 0: out += plan(k, c, depth + 1)
    return out

# 1. Plan (saved, so a rerun resumes)
if os.path.exists('kiva_slices/extra_plan.json'):
    slices = json.load(open('kiva_slices/extra_plan.json'))
    for f, n in slices:
        for k in ['term'] + [d for d, _, _ in DIMS]:
            if k in f: f[k] = tuple(f[k])
else:
    slices = []
    for b in BIG: slices += plan(b)
    json.dump(slices, open('kiva_slices/extra_plan.json', 'w'))
print(f'\n{len(slices)} extra slices covering {sum(n for _, n in slices):,} loans\n')

# 2. Download
for i, (f, n) in enumerate(slices):
    path = f'kiva_slices/extra_{i:03d}.csv'
    if os.path.exists(path): continue
    rows = []
    for off in range(0, min(n, CAP), 100):
        q = f'{{ lend {{ loans(filters: {fstr(f)}, limit: 100, offset: {off}) {{ values {{ {FIELDS} }} }} }} }}'
        rows += gql(q)['lend']['loans']['values']
        time.sleep(0.2)
    pd.json_normalize(rows).to_csv(path, index=False)
    print(f'extra slice {i+1}/{len(slices)} done: {len(rows):,} loans')

# 3. Combine everything
files = sorted(glob.glob('kiva_slices/slice_*.csv') + glob.glob('kiva_slices/extra_*.csv'))
df = pd.concat([pd.read_csv(p) for p in files]).drop_duplicates('id')
df.to_csv('kiva_kenya_api.csv', index=False)
print(f'\nDone: {len(df):,} unique Kenya loans saved to kiva_kenya_api.csv')

gender=female, isGroup=False, term=(14, 14) -> 152,492
  gender=female, isGroup=False, term=(14, 14), sector=(1, 60) -> 152,492
    gender=female, isGroup=False, term=(14, 14), sector=(1, 30) -> 152,492
       !! this split covers 152,487 of 152,492
      gender=female, isGroup=False, term=(14, 14), sector=(1, 15) -> 147,275
         !! this split covers 147,256 of 147,275
        gender=female, isGroup=False, term=(14, 14), sector=(1, 8) -> 117,918
           !! this split covers 117,892 of 117,918
          gender=female, isGroup=False, term=(14, 14), sector=(1, 4) -> 103,251
             !! this split covers 103,230 of 103,251
            gender=female, isGroup=False, term=(14, 14), sector=(1, 2) -> 98,073
              gender=female, isGroup=False, term=(14, 14), sector=(1, 1) -> 98,073
                gender=female, isGroup=False, term=(14, 14), sector=(1, 1), activity=(1, 400) -> 98,073
                  gender=female, isGroup=False, term=(14, 14), sector=(1, 1), activity=(1, 200